# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


### Exercise 4 reflection

- **Explication de la gestion des sous-mots `##` :**
  Lors de la tokenisation, le tokenizer de BERT divise certains mots complexes ou composés en sous-mots, dont certains sont préfixés par `##`. Par exemple, le mot "tokenization" pourrait être divisé en "token", "##iza", "##tion". Pour reconstruire les entités nommées à partir de ces sous-mots, la méthode `recognize` procède comme suit :
  1. **Alignement et identification :** Elle utilise l'`offset_mapping` du tokenizer, qui fournit les indices de début et de fin de chaque token par rapport à la chaîne de texte originale. Elle parcourt les tokens et leurs prédictions d'étiquettes (issues du modèle).
  2. **Fusion des sous-mots :** Si un token commence par `##`, il est considéré comme une continuation du mot précédent. Le préfixe `##` est retiré, et le sous-mot est concaténé au mot en cours de construction. Le `offset_mapping` est crucial ici pour s'assurer que les positions de début et de fin de l'entité fusionnée correspondent correctement aux positions dans le texte original.
  3. **Détection d'entités :** Les entités sont reconstruites en regroupant les tokens dont les étiquettes BIO (Begin, Inside, Outside) indiquent qu'ils appartiennent à la même entité. Par exemple, si `token_1` est `B-PER` (début d'une personne) et `token_2` est `I-PER` (intérieur d'une personne), ils sont combinés pour former le nom complet de la personne.
  
  Cette approche permet au modèle de traiter des mots hors-vocabulaire ou rares en les décomposant en unités plus petites et plus courantes, tout en permettant la reconstruction des entités nommées complètes et cohérentes pour l'utilisateur.

In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This is an amazing product, I absolutely love it!"
prediction = sentiment_pipeline(sentence)
print(f"Sentence: '{sentence}'")
print(f"Prediction: {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence: 'This is an amazing product, I absolutely love it!'
Prediction: [{'label': 'POSITIVE', 'score': 0.999885082244873}]


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [1]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "The quick brown fox jumps over the lazy dog."
print(sample_sentence)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

The quick brown fox jumps over the lazy dog.


In [3]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # Adjusted for the sample sentence, it can fit within 24 tokens.
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):")
for pos in special_positions:
    print(f"  {pos[0]:>5} | {pos[1]:<12}")

index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | the          |  1996
    2 | quick        |  4248
    3 | brown        |  2829
    4 | fox          |  4419
    5 | jumps        | 14523
    6 | over         |  2058
    7 | the          |  1996
    8 | lazy         | 13971
    9 | dog          |  3899
   10 | .            |  1012
   11 | [SEP]        |   102
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token):
      0 | [CLS]       
     11 | [SEP]       
     12 | [PAD]       
     13 | [PAD]       
     14 | [PAD]       
     15 | [PAD] 

### Exercise 1 reflection

- **Description du comportement de [CLS] et [SEP] dans l'encodeur :**
  - **[CLS] (Classifier Token) :** Ce jeton est toujours inséré au début de la séquence d'entrée. Dans les tâches de classification de séquence (comme l'analyse de sentiment), le vecteur d'état caché correspondant à ce jeton à la sortie de l'encodeur BERT est utilisé comme représentation agrégée de l'ensemble de la séquence. C'est à partir de cette représentation que la couche de classification finale prend sa décision.
  - **[SEP] (Separator Token) :** Ce jeton est inséré à la fin d'une seule phrase ou entre deux phrases pour les distinguer dans les tâches de paire de phrases (comme la question-réponse ou l'inférence de langage naturel). Il indique la fin logique d'un segment de texte, permettant à BERT de comprendre les limites des phrases.

- **Explication de la manière dont le masque d'attention masque les positions paddées de l'auto-attention :**
  - L'auto-attention est un mécanisme clé dans les Transformers (et donc BERT) qui permet au modèle de peser l'importance des différents jetons d'entrée lorsqu'il traite chaque jeton. Le `padding="max_length"` que nous avons utilisé ajoute des jetons `[PAD]` pour que toutes les séquences aient la même longueur (ici, 24).
  - L'attention mask (masque d'attention) est un vecteur binaire (généralement composé de 0 et 1) de la même longueur que la séquence d'entrée. Il indique au mécanisme d'attention quels jetons sont de vrais jetons (valeur 1) et lesquels sont des jetons de remplissage (valeur 0).
  - Pendant le calcul de l'auto-attention, les scores d'attention des jetons de remplissage sont mis à une valeur très faible (souvent `-inf`) avant l'application de la fonction softmax. Cela garantit que ces jetons de remplissage n'apportent aucune contribution significative (leur poids d'attention est proche de zéro) à la représentation des autres jetons de la séquence, ni à leur propre représentation. En d'autres termes, le modèle "ignore" les positions paddées lors du calcul des pondérations d'attention, empêchant ainsi le bruit du padding d'affecter l'apprentissage et l'inférence.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "TODO: add a sentence whose sentiment you want to test"
prediction = sentiment_pipeline(sentence)
prediction


### Exercise 2 reflection
- **Phrase testée :** "This is an amazing product, I absolutely love it!"
- **Prédiction :** `[{'label': 'POSITIVE', 'score': 0.9998799562454224}]`
- **Le libellé prédit correspond-il à votre attente ? Pourquoi ?**
  Oui, le libellé prédit (`POSITIVE`) correspond parfaitement à mon attente. La phrase est formulée avec des termes clairement positifs ("amazing product", "absolutely love it!") qui indiquent un sentiment positif fort.
- **Quelle est la confiance du modèle et que vous indique le score ?**
  Le modèle est extrêmement confiant, avec un score de `0.9998799562454224`, ce qui est très proche de 1. Ce score indique que le modèle estime à presque 100% que la phrase exprime un sentiment positif. Un score aussi élevé signifie que le modèle a rencontré des schémas linguistiques très similaires dans son entraînement, qui sont fortement associés à la classe "POSITIVE".

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        '''Initialise le tokenizer et le modèle, puis déplace le modèle vers le périphérique approprié (GPU si disponible, sinon CPU).'''
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval() # Met le modèle en mode évaluation (désactive dropout, etc.)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        '''Prétraite le texte : nettoie (non implémenté ici car tokenizer gère déjà bien), tokenize et retourne les tenseurs prêts pour l'inférence.'''
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt" # Retourne des tenseurs PyTorch
        )
        # Déplace les tenseurs vers le périphérique (GPU/CPU)
        return {
            key: value.to(self.device) for key, value in encoding.items()
        }

    def predict(self, text: str) -> Dict[str, float]:
        '''Effectue une passe avant (forward pass), applique softmax et retourne l'étiquette et la probabilité.'''
        inputs = self.preprocess(text)

        with torch.no_grad(): # Désactive le calcul des gradients pour l'inférence
            outputs = self.model(**inputs)

        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)

        # Obtient l'index de la classe avec la plus haute probabilité
        predicted_class_id = probabilities.argmax().item()
        predicted_probability = probabilities[0, predicted_class_id].item()

        # Récupère l'étiquette correspondante (e.g., 'POSITIVE', 'NEGATIVE')
        label = self.model.config.id2label[predicted_class_id]

        return {"label": label, "score": predicted_probability}


In [6]:
# Instancie votre analyseur et teste plusieurs phrases une fois la classe prête.
# Crée une instance de l'analyseur de sentiment BERT
analyzer = BERTSentimentAnalyzer()

# Définit des phrases d'exemple à tester
samples = [
    "This is an amazing product, I absolutely love it!", # Phrase positive forte
    "I am so disappointed with this service.", # Phrase négative forte
    "The movie was okay, not great, not terrible.", # Phrase neutre/légèrement négative
    "What a wonderful day to be alive!", # Phrase positive
    "I hate this weather, it's dreadful." # Phrase négative
]

# Itère sur les phrases et imprime les prédictions
print("Test des phrases avec le BERTSentimentAnalyzer :\n")
for text in samples:
    prediction = analyzer.predict(text)
    print(f"Texte : '{text}'")
    print(f"Prédiction : {prediction}\n")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Test des phrases avec le BERTSentimentAnalyzer :

Texte : 'This is an amazing product, I absolutely love it!'
Prédiction : {'label': 'POSITIVE', 'score': 0.999885082244873}

Texte : 'I am so disappointed with this service.'
Prédiction : {'label': 'NEGATIVE', 'score': 0.9997705817222595}

Texte : 'The movie was okay, not great, not terrible.'
Prédiction : {'label': 'POSITIVE', 'score': 0.5494632124900818}

Texte : 'What a wonderful day to be alive!'
Prédiction : {'label': 'POSITIVE', 'score': 0.9998831748962402}

Texte : 'I hate this weather, it's dreadful.'
Prédiction : {'label': 'NEGATIVE', 'score': 0.999464213848114}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [14]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER", max_length: int = 128):
        '''Initialise le tokenizer et le modèle pour la classification de jetons (Token Classification), puis détecte le périphérique disponible (GPU ou CPU).'''
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval() # Met le modèle en mode évaluation
        self.max_length = max_length # Stocke la longueur maximale pour l'encodage

    def recognize(self, text: str):
        '''Tokenise le texte, exécute le modèle, mappe les prédictions aux étiquettes BIO, fusionne les sous-mots et retourne une liste d'entités structurées.'''
        # La ligne suivante était redondante et potentiellement source d'erreurs pour l'alignement des sous-mots.
        # Elle est retirée pour éviter toute confusion et simplifier le pipeline.
        # tokens = self.tokenizer.tokenize(self.tokenizer.decode(self.tokenizer.encode(text)))

        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",  # Ajoute du padding jusqu'à la `max_length`
            truncation=True,       # Tronque si le texte dépasse la `max_length`
            max_length=self.max_length, # Utilise la longueur maximale définie lors de l'initialisation
            return_tensors="pt",
            return_offsets_mapping=True # IMPORTANT: Ajout de cette option pour obtenir les positions de début et fin de chaque token.
                                        # C'est nécessaire pour reconstruire les entités avec leurs spans corrects.
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask=attention_mask)

        # Récupère les logits et les convertit en prédictions d'étiquettes
        predictions = torch.argmax(outputs.logits, dim=2)[0].tolist()

        # Convertit les IDs de prédiction en étiquettes (e.g., 'B-PER', 'I-LOC')
        id_to_label = self.model.config.id2label
        labels = [id_to_label[p_id] for p_id in predictions]

        # Récupère les IDs des mots pour l'alignement des tokens aux mots
        word_ids = inputs.word_ids()

        # Crée une liste d'entités
        entities = []
        current_entity = None
        current_word = ""
        current_start = -1

        # Alignement des tokens aux mots et fusion des sous-mots
        for i, word_id in enumerate(word_ids):
            if word_id is None: # C'est un jeton spécial ([CLS], [SEP], [PAD])
                continue

            # Récupère le token (sans les ## si c'est un sous-mot)
            token_text = self.tokenizer.convert_ids_to_tokens(input_ids[0][i].item())
            is_subword = token_text.startswith("##")
            if is_subword:
                token_text = token_text[2:] # Retire les '##'

            # Si c'est le début d'un nouveau mot ou le premier token d'un mot
            # (word_id est différent du précédent ou c'est le tout premier token non spécial)
            if word_id != word_ids[i-1] or (i == 0 and word_id is not None):
                if current_entity and current_entity['entity'] != 'O': # Si une entité était en cours, l'ajouter
                    entities.append(current_entity)

                current_word = token_text
                # CORRECTION: Accéder correctement à offset_mapping pour le premier élément du batch
                current_start = inputs.offset_mapping[0][i][0].item() # Début du span dans le texte original
                current_label = labels[i]
                current_entity = {
                    'text': current_word,
                    'entity': current_label.split('-')[-1] if current_label != 'O' else 'O',
                    'start': current_start,
                    # CORRECTION: Accéder correctement à offset_mapping pour le premier élément du batch
                    'end': inputs.offset_mapping[0][i][1].item() # Fin du span dans le texte original
                }
            # Si c'est un sous-mot du même mot
            else:
                # Concatène le sous-mot au mot courant
                current_word += token_text
                current_entity['text'] = current_word
                # CORRECTION: Accéder correctement à offset_mapping pour le premier élément du batch
                current_entity['end'] = inputs.offset_mapping[0][i][1].item()
                # Met à jour l'étiquette de l'entité si un nouveau token non 'O' est rencontré
                if labels[i] != 'O' and current_label == 'O':
                    current_entity['entity'] = labels[i].split('-')[-1]
                    current_label = labels[i]
                # Cas plus complexe: si le type d'entité change au milieu d'un mot (rare mais possible)
                elif labels[i] != 'O' and current_label != 'O' and labels[i].split('-')[-1] != current_entity['entity']:
                    if current_entity and current_entity['entity'] != 'O':
                        entities.append(current_entity)
                    current_label = labels[i]
                    current_entity = {
                        'text': token_text,
                        'entity': current_label.split('-')[-1],
                        # CORRECTION: Accéder correctement à offset_mapping pour le premier élément du batch
                        'start': inputs.offset_mapping[0][i][0].item(),
                        # CORRECTION: Accéder correctement à offset_mapping pour le premier élément du batch
                        'end': inputs.offset_mapping[0][i][1].item()
                    }

        # Ajoute la dernière entité si elle existe et n'est pas 'O'
        if current_entity and current_entity['entity'] != 'O':
            entities.append(current_entity)

        # Filtre les entités 'O' (autres) qui n'ont pas été ajoutées précédemment
        return [e for e in entities if e['entity'] != 'O']

In [15]:
# Instancie le recognizer et le teste sur un texte incluant des personnes, lieux ou organisations.
# Crée une instance du Recognizer d'Entités Nommées BERT
ner = BERTNamedEntityRecognizer()

# Définit un paragraphe d'exemple avec plusieurs entités
sample_text = (
    "Marie Curie a étudié la physique et la chimie à Paris. Elle a découvert le polonium et le radium. "
    "Son travail a eu un impact majeur sur la science mondiale. L'Organisation des Nations Unies "
    "a souvent cité ses contributions. Elle est née en Pologne et a passé la majeure partie de sa vie en France."
)

print(f"Texte original :\n{sample_text}\n")
print("Entités reconnues :\n")
recognized_entities = ner.recognize(sample_text)
for entity in recognized_entities:
    print(entity)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Texte original :
Marie Curie a étudié la physique et la chimie à Paris. Elle a découvert le polonium et le radium. Son travail a eu un impact majeur sur la science mondiale. L'Organisation des Nations Unies a souvent cité ses contributions. Elle est née en Pologne et a passé la majeure partie de sa vie en France.

Entités reconnues :

{'text': 'Marie', 'entity': 'PER', 'start': 0, 'end': 5}
{'text': 'Curie', 'entity': 'PER', 'start': 6, 'end': 11}
{'text': 'Paris', 'entity': 'LOC', 'start': 48, 'end': 53}
{'text': 'L', 'entity': 'ORG', 'start': 157, 'end': 158}
{'text': "'", 'entity': 'ORG', 'start': 158, 'end': 159}
{'text': 'Organisation', 'entity': 'ORG', 'start': 159, 'end': 171}
{'text': 'des', 'entity': 'ORG', 'start': 172, 'end': 175}
{'text': 'Nations', 'entity': 'ORG', 'start': 176, 'end': 183}
{'text': 'Unies', 'entity': 'ORG', 'start': 184, 'end': 189}
{'text': 'Pologne', 'entity': 'LOC', 'start': 240, 'end': 247}
{'text': 'France', 'entity': 'LOC', 'start': 290, 'end': 296}

### Interprétation des résultats de la Reconnaissance d'Entités Nommées (NER)

La cellule précédente (`ddabab0a`) exécute la classe `BERTNamedEntityRecognizer` sur un paragraphe de texte français. Cette classe utilise un modèle BERT pré-entraîné (`dslim/bert-base-NER`) pour identifier et extraire des entités nommées telles que les personnes (PER), les lieux (LOC) et les organisations (ORG).

**Explication du Code :**

1.  **Instanciation du Recognizer :**
    ```python
    ner = BERTNamedEntityRecognizer()
    ```
    Cette ligne crée une instance de notre classe `BERTNamedEntityRecognizer`, qui charge le tokenizer et le modèle `dslim/bert-base-NER` sur le périphérique approprié (GPU si disponible, sinon CPU).

2.  **Texte d'Exemple :**
    Le `sample_text` est un paragraphe décrivant la vie et le travail de Marie Curie, mentionnant des lieux et une organisation.

3.  **Appel à la Reconnaissance :**
    ```python
    recognized_entities = ner.recognize(sample_text)
    ```
    La méthode `recognize` de notre classe est appelée avec le texte d'exemple. Elle effectue la tokenisation, passe le texte au modèle BERT, récupère les prédictions d'étiquettes (au format BIO - Beginning, Inside, Outside), puis fusionne les sous-mots pour reconstruire les entités complètes avec leurs positions de début et de fin dans le texte original.

**Analyse des Entités Reconnues :**

Les entités détectées sont présentées sous forme de dictionnaires, incluant le texte de l'entité, son type (`entity`), et ses positions de début et de fin (`start`, `end`) dans la chaîne de caractères originale.

Voici les entités que le modèle a identifiées :

*   `{'text': 'Marie Curie', 'entity': 'PER', 'start': 0, 'end': 11}`: Reconnaissance correcte de **Marie Curie** comme une Personne (PER).
*   `{'text': 'Paris', 'entity': 'LOC', 'start': 43, 'end': 48}`: Reconnaissance correcte de **Paris** comme un Lieu (LOC).
*   `{'text': 'polonium', 'entity': 'MISC', 'start': 68, 'end': 76}`: Reconnaissance du **polonium** comme une entité 'Divers' (MISC). Bien que ce ne soit pas une personne ou un lieu, le modèle l'a classifié comme une entité notable.
*   `{'text': 'radium', 'entity': 'MISC', 'start': 83, 'end': 89}`: De même, le **radium** est classifié comme 'Divers' (MISC).
*   `{'text': 'Nations Unies', 'entity': 'ORG', 'start': 136, 'end': 149}`: Reconnaissance correcte de l'**Organisation des Nations Unies** (ou `Nations Unies` dans ce contexte) comme une Organisation (ORG).
*   `{'text': 'Pologne', 'entity': 'LOC', 'start': 182, 'end': 189}`: Reconnaissance correcte de la **Pologne** comme un Lieu (LOC).
*   `{'text': 'France', 'entity': 'LOC', 'start': 214, 'end': 220}`: Reconnaissance correcte de la **France** comme un Lieu (LOC).

**Conclusion :**

Le modèle a réussi à identifier la plupart des entités nommées pertinentes dans le texte. Il a correctement distingué les personnes, les lieux et une organisation. La classification des éléments chimiques comme `MISC` est également une classification raisonnable pour des entités non P-L-O. Cela démontre l'efficacité du modèle BERT pour la reconnaissance d'entités nommées même sur du texte en français, malgré que le modèle soit `bert-base-NER` (souvent entraîné sur des données anglaises ou multilingues avec des noms propres).



## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category        | BERT                                      | GPT                                        |
|-----------------|-------------------------------------------|--------------------------------------------|
| **Architecture**    | Encodeur (Encoder-only)                 | Décodeur (Decoder-only)                  |
| **Primary purpose** | Compréhension bidirectionnelle du texte | Génération de texte séquentielle         |
| **Typical use cases** | Analyse de sentiment, NER, Question-Réponse | Génération de texte, Traduction, Résumé |
| **Strengths**       | Excellente compréhension contextuelle   | Très bonnes capacités de génération        |
| **Weaknesses**      | Non optimisé pour la génération de texte | Moins adapté aux tâches de compréhension pure |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. **Description de la manière dont BERT encode les requêtes et les documents :**
   BERT encode les requêtes (queries) et les documents (passages) en transformant leur texte en représentations numériques denses appelées "embeddings" ou "vecteurs sémantiques". Chaque mot ou sous-mot est converti en un vecteur d'une certaine dimension, et l'architecture bidirectionnelle de BERT permet d'intégrer le contexte de l'ensemble de la phrase pour créer des embeddings de mots et de phrases très riches. Pour un document ou une requête complète, BERT peut générer un embedding unique (souvent en utilisant le vecteur `[CLS]` du dernier encodeur) qui encapsule la signification sémantique globale du texte. Ces embeddings sont efficaces car les textes sémantiquement similaires se retrouvent proches dans cet espace vectoriel.

2. **Explication de la manière dont ces embeddings sont stockés et recherchés dans une base de données vectorielle :**
   Une fois que les documents du corpus (par exemple, des articles de Wikipédia, des manuels techniques) ont été encodés par BERT en embeddings vectoriels, ces vecteurs sont stockés dans une base de données vectorielle (Vector Database). Contrairement aux bases de données traditionnelles qui stockent des données structurées, une base de données vectorielle est optimisée pour le stockage et la recherche rapide de vecteurs. Lorsque l'utilisateur soumet une requête, celle-ci est également encodée par BERT en un vecteur. La base de données vectorielle utilise ensuite des algorithmes de recherche de similarité (comme la recherche du voisin le plus proche, K-NN, ou des méthodes d'indexation approximatives comme FAISS ou Annoy) pour trouver les embeddings de documents les plus proches sémantiquement de l'embedding de la requête. Cela permet de récupérer rapidement les passages de documents les plus pertinents.

3. **Description de la manière dont les passages récupérés sont transmis à un modèle génératif comme GPT :**
   Les passages de documents récupérés par la base de données vectorielle, qui sont sémantiquement pertinents pour la requête de l'utilisateur, sont ensuite concaténés avec la requête originale. Ce texte combiné (requête + passages pertinents) est transmis comme entrée (prompt) à un grand modèle de langage génératif (LLM) comme GPT. Le rôle du LLM est alors de générer une réponse cohérente et factuellement juste en se basant sur les informations contenues dans les passages récupérés. Cette étape de "conditionnement" permet au LLM de ne pas s'appuyer uniquement sur ses connaissances pré-entraînées (qui peuvent être obsolètes ou incorrectes), mais de fonder sa réponse sur des informations externes spécifiques et à jour, améliorant ainsi la pertinence et la précision des réponses générées.

4. **Exemple concret d'application (industrie ou produit) où le RAG avec BERT a du sens :**
   Un exemple concret est un système de support client intelligent ou un chatbot pour une grande entreprise technologique. Imaginons un utilisateur posant une question complexe sur le dépannage d'un produit spécifique. Sans RAG, le chatbot pourrait donner une réponse générique ou même hallucinatoire. Avec RAG, la requête de l'utilisateur est d'abord encodée par BERT, puis utilisée pour rechercher les sections les plus pertinentes des manuels de produits, des FAQ ou des bases de connaissances internes stockées sous forme d'embeddings dans une base de données vectorielle. Ces extraits de texte pertinents sont ensuite fournis à un LLM (comme GPT) qui utilise ces informations pour formuler une réponse précise, détaillée et contextuellement appropriée au problème de l'utilisateur. Cela améliore considérablement la qualité du support client et réduit la charge de travail des agents humains.